In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)

column_types = {
    "id": "Int64",
    "osm_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
}

In [2]:
data_gdynia = pd.read_csv(
    "../data/processed/fixed/srodmiescie.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
    parse_dates=["date_of_checking"],
)
data_gdynia

,osm_id,addr_street,addr_number,ground_floor_group,ground_floor_function,upper_floor_group,upper_floor_function,number_of_storeys,is_mixed_use,build_year_est,arch_style,condition,max_height,is_heritage,source,confidence,date_of_checking,number_of_functions
id,,,,,,,,,,,,,,,,,,
1,127184291,Świętego Piotra,8,NaN,hotel,NaN,Hotel,7,<NA>,2000,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27,1
2,890729731,Świętego Piotra,6D,NaN,mieszkaniowa wielorodzinna,NaN,Mieszkaniowa Wielorodzinna,5,<NA>,2020,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27,1
3,890292040,Węglowa,22,NaN,centrum handlowe,NaN,Mieszkaniowa Wielorodzinna,17,<NA>,2020,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27,1
4,890729732,Świętego Piotra,6,NaN,mieszkaniowa wielorodzinna,NaN,Mieszkaniowa Wielorodzinna,17,<NA>,2020,współczesność,3,NaN,NaN,google.com/maps,3,2026-09-27,1
5,136885205,Tadeusza Wendy,15,NaN,centrum medyczne,NaN,Centrum Medyczne,6,<NA>,2000,współczesność,3,NaN,NaN,google.com/maps,2,2026-09-27,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1237,126180359,Władysława IV,61,NaN,"laboratorium medyczne,piekarnia,kiosk",NaN,Mieszkaniowa Wielorodzinna,7,<NA>,1940,modernizm,3,NaN,NaN,google.com/maps,2,2026-09-30,3
1238,126173503,Świętojańska,122,NaN,"apteka,handel zdrowie",NaN,Biurowa,6,<NA>,1930,modernizm,2,NaN,NaN,google.com/maps,2,2026-09-30,2
1239,260676369,Aleja Marszałka Józefa Piłsudskiego,56,NaN,mieszkaniowa wielorodzinna,NaN,Mieszkaniowa Wielorodzinna,7,<NA>,1930,modernizm,2,NaN,NaN,google.com/maps,2,2026-09-30,1


In [3]:
data_gdynia_long = pd.read_csv(
    "../data/processed/fixed/srodmiescie_long.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
)
data_gdynia_long.head()

,addr_street,ground_floor_function
id,,
1,Świętego Piotra,hotel
2,Świętego Piotra,mieszkaniowa wielorodzinna
3,Węglowa,centrum handlowe
4,Świętego Piotra,mieszkaniowa wielorodzinna
5,Tadeusza Wendy,centrum medyczne


1.

In [4]:
mean_num_storeys_street_agg = data_gdynia.groupby("addr_street")[
    "number_of_storeys"
].agg("mean")
mean_num_storeys_street_trans = data_gdynia.groupby("addr_street")[
    "number_of_storeys"
].transform("mean")

In [5]:
mean_num_storeys_street_agg.head()

addr_street
10 Lutego                                   4.0
3 Maja                                 4.269231
Aleja Jana Pawła II                    1.533333
Aleja Marszałka Józefa Piłsudskiego    6.333333
Antoniego Abrahama                       4.6625
Name: number_of_storeys, dtype: Float64

In [6]:
mean_num_storeys_street_trans

id
1         4.0625
2         4.0625
3           17.0
4         4.0625
5            4.0
          ...   
1237    6.195122
1238        5.08
1239    6.333333
1240    6.333333
1241    6.333333
Name: number_of_storeys, Length: 1105, dtype: Float64

In [7]:
assert len(mean_num_storeys_street_trans) == len(data_gdynia)
assert mean_num_storeys_street_trans.isna().equals(data_gdynia["addr_street"].isna())
assert mean_num_storeys_street_trans.index.equals(data_gdynia.index)

2.

In [8]:
data_gdynia = data_gdynia.assign(
    street_mean=lambda d: d.groupby("addr_street")["number_of_storeys"].transform(
        "mean"
    ),
    storeys_vs_street=lambda d: d["number_of_storeys"] - d["street_mean"],
)

In [9]:
data_gdynia.loc[
    data_gdynia["storeys_vs_street"].abs().nlargest(10).index,
    ["addr_street", "number_of_storeys", "street_mean", "storeys_vs_street"],
]

,addr_street,number_of_storeys,street_mean,storeys_vs_street
id,,,,
4,Świętego Piotra,17,4.0625,12.9375
815,Armii Krajowej,13,4.638889,8.361111
813,3 Maja,11,4.269231,6.730769
849,3 Maja,11,4.269231,6.730769
232,Plac Konstytucji,9,2.636364,6.363636
399,Jerzego Waszyngtona,11,4.827586,6.172414
575,Mściwoja,1,6.9,-5.9
753,Stefana Batorego,1,6.6,-5.6
234,Plac Kaszubski,11,5.571429,5.428571


In [10]:
assert np.isclose(
    data_gdynia.groupby("addr_street")["storeys_vs_street"].agg("mean"), 0
).all()

3.

In [11]:
style_share_per_street = (
    data_gdynia.groupby(["addr_street", "arch_style"])
    .agg(num_build=("osm_id", "size"))
    .assign(
        street_total=lambda d: d.groupby("addr_street")["num_build"].transform("sum"),
        share=lambda d: d["num_build"] / d["street_total"],
    )
)

In [12]:
style_share_per_street.head(10)

num_build  street_total  \
addr_street                         arch_style                               
10 Lutego                           klasycyzm             15            48   
                                    modernizm             15            48   
                                    współczesność         18            48   
3 Maja                              klasycyzm              3            26   
                                    modernizm             14            26   
                                    współczesność          9            26   
Aleja Jana Pawła II                 modernizm              4            15   
                                    współczesność         11            15   
Aleja Marszałka Józefa Piłsudskiego modernizm              3             3   
Antoniego Abrahama                  klasycyzm              5            80   

                                                      share  
addr_street                         arch_style               
10 Lutego                           klasycyzm      0.312500  
                                    modernizm      0.312500  
                                    współczesność  0.375000  
3 Maja                              klasycyzm      0.115385  
                                    modernizm      0.538462  
                                    współczesność  0.346154  
Aleja Jana Pawła II                 modernizm      0.266667  
                                    współczesność  0.733333  
Aleja Marszałka Józefa Piłsudskiego modernizm      1.000000  
Antoniego Abrahama                  klasycyzm      0.062500

In [13]:
sum_shares_per_street = style_share_per_street.groupby("addr_street").agg(
    sum_shares=("share", "sum")
)
assert np.isclose(sum_shares_per_street, 1).all()

5.

In [14]:
data_gdynia = data_gdynia.assign(
    mean_street=lambda d: d.groupby("addr_street")["number_of_storeys"].transform(
        "mean"
    ),
    std_street=lambda d: d.groupby("addr_street")["number_of_storeys"].transform(
        np.std
    ),
    z_in_street=lambda d: (d["number_of_storeys"] - d["mean_street"]) / d["std_street"],
)

In [15]:
data_gdynia.loc[
    data_gdynia["z_in_street"].abs() > 2,
    ["number_of_storeys", "mean_street", "std_street", "z_in_street"],
]

,number_of_storeys,mean_street,std_street,z_in_street
id,,,,
4,17,4.0625,3.928561,3.29319
69,2,1.111111,0.314270,2.828427
86,8,3.121951,2.120689,2.300219
127,7,2.48,1.676186,2.696598
131,6,2.48,1.676186,2.100005
232,9,2.636364,2.532434,2.512854
234,11,5.571429,2.470231,2.197597
399,11,4.827586,2.817054,2.191089
480,8,5.909091,0.792527,2.638281


In [16]:
assert np.isclose(data_gdynia.groupby("addr_street")["z_in_street"].agg("sum"), 0).all()

6.

In [17]:
data_gdynia_rank = data_gdynia.assign(
    rank_min=lambda d: d.groupby("addr_street")["number_of_storeys"].rank(
        ascending=False, method="min"
    ),
    rank_dense=lambda d: d.groupby("addr_street")["number_of_storeys"].rank(
        ascending=False, method="dense"
    ),
    rank_first=lambda d: d.groupby("addr_street")["number_of_storeys"].rank(
        ascending=False, method="first"
    ),
)

data_gdynia_rank.loc[
    :, ["addr_street", "number_of_storeys", "rank_min", "rank_dense", "rank_first"]
]

,addr_street,number_of_storeys,rank_min,rank_dense,rank_first
id,,,,,
1,Świętego Piotra,7,2.0,2.0,2.0
2,Świętego Piotra,5,3.0,3.0,3.0
3,Węglowa,17,1.0,1.0,1.0
4,Świętego Piotra,17,1.0,1.0,1.0
5,Tadeusza Wendy,6,3.0,3.0,3.0
...,...,...,...,...,...
1237,Władysława IV,7,11.0,4.0,17.0
1238,Świętojańska,6,10.0,4.0,81.0
1239,Aleja Marszałka Józefa Piłsudskiego,7,1.0,1.0,1.0


In [18]:
assert (
    data_gdynia_rank.groupby("addr_street")["rank_first"].max()
    == data_gdynia_rank.groupby("addr_street")["rank_first"].size()
).all()

7.

In [ ]:
data_gdynia = data_gdynia.assign(
    order_in_street=lambda d: (
        d.sort_values(["date_of_checking", "id"]).groupby("addr_street").cumcount() + 1
    )
)

In [ ]:
data_gdynia.loc[:, ["addr_street", "date_of_checking", "order_in_street"]]

,addr_street,date_of_checking,order_in_street
id,,,
1,Świętego Piotra,2026-09-27,1.0
2,Świętego Piotra,2026-09-27,2.0
3,Węglowa,2026-09-27,1.0
4,Świętego Piotra,2026-09-27,3.0
5,Tadeusza Wendy,2026-09-27,1.0
...,...,...,...
1237,Władysława IV,2026-09-30,41.0
1238,Świętojańska,2026-09-30,125.0
1239,Aleja Marszałka Józefa Piłsudskiego,2026-09-30,1.0


In [ ]:
assert (
    data_gdynia.groupby("addr_street")["order_in_street"].max()
    == data_gdynia.groupby("addr_street")["osm_id"].agg("size")
).all()

8.

In [ ]:
data_gdynia_top_2_height = (
    data_gdynia.sort_values(["number_of_storeys", "id"], ascending=[False, True])
    .groupby("addr_street")[["addr_street", "number_of_storeys"]]
    .head(2)
    .sort_values(["addr_street", "number_of_storeys"], ascending=[True, False])
)
data_gdynia_top_2_height.head(20)

,addr_street,number_of_storeys
id,,
645,10 Lutego,8
650,10 Lutego,7
813,3 Maja,11
849,3 Maja,11
785,Aleja Jana Pawła II,5
774,Aleja Jana Pawła II,4
1239,Aleja Marszałka Józefa Piłsudskiego,7
1240,Aleja Marszałka Józefa Piłsudskiego,7
916,Antoniego Abrahama,7


In [ ]:
assert (data_gdynia_top_2_height.groupby("addr_street").size() <= 2).all()

9.

In [ ]:
data_gdynia_most_frequent_style = data_gdynia.groupby(
    ["addr_street", "arch_style"]
).agg(num_build=("osm_id", "size"))

In [ ]:
data_gdynia_most_frequent_style2 = (
    data_gdynia_most_frequent_style.sort_values(
        ["num_build", "arch_style"], ascending=[False, True]
    )
    .groupby("addr_street")
    .head(2)
    .sort_values(
        ["addr_street", "num_build", "arch_style"], ascending=[True, False, True]
    )
)
data_gdynia_most_frequent_style2.head(20)

num_build
addr_street                           arch_style              
10 Lutego                             współczesność         18
                                      klasycyzm             15
3 Maja                                modernizm             14
                                      współczesność          9
Aleja Jana Pawła II                   współczesność         11
                                      modernizm              4
Aleja Marszałka Józefa Piłsudskiego   modernizm              3
Antoniego Abrahama                    modernizm             55
                                      współczesność         20
Antoniego Hryniewickiego              współczesność         15
                                      modernizm             10
Arkadiusza Rybickiego                 współczesność          9
Armii Krajowej                        modernizm             23
                                      współczesność          8
Dworcowa                              modernizm              3
                                      współczesność          2
Franciszka Żwirki i Stanisława Wigury modernizm              9
                                      współczesność          1
Generała Józefa Bema                  modernizm             10
                                      współczesność          1

In [ ]:
assert (data_gdynia_most_frequent_style2.groupby("addr_street").size() <= 2).all()

In [ ]:
data_gdynia_most_frequent_style["rank"] = data_gdynia_most_frequent_style.groupby(
    "addr_street"
)["num_build"].rank(method="dense", ascending=False)

In [ ]:
mask = data_gdynia_most_frequent_style["rank"] <= 2
data_gdynia_most_frequent_style_rank = data_gdynia_most_frequent_style.loc[
    mask
].sort_values(["addr_street", "rank"], ascending=[True, True])

In [ ]:
assert data_gdynia_most_frequent_style_rank["rank"].nunique() == 2
assert (data_gdynia_most_frequent_style_rank["rank"] <= 2).all()

In [ ]:
data_gdynia_most_frequent_style_rank.loc[
    (data_gdynia_most_frequent_style_rank.groupby("addr_street").transform("size") > 2)
]

num_build  rank
addr_street      arch_style                    
10 Lutego        współczesność         18   1.0
                 klasycyzm             15   2.0
                 modernizm             15   2.0
Plac Grunwaldzki modernizm              4   1.0
                 klasycyzm              2   2.0
                 współczesność          2   2.0
Stefana Batorego modernizm              4   1.0
                 postmodernizm          3   2.0
                 współczesność          3   2.0
Świętojańska     modernizm            107   1.0
                 klasycyzm              9   2.0
                 współczesność          9   2.0

10.

In [ ]:
data_gdynia_sum_build_style = data_gdynia.groupby("arch_style").agg(
    num_style=("osm_id", "size")
)
data_gdynia_sum_build_style

,num_style
arch_style,
klasycyzm,59
modernizm,530
postmodernizm,7
tradycjonalizm,8
współczesność,501


In [ ]:
assert len(data_gdynia_sum_build_style) == data_gdynia["arch_style"].nunique()

In [ ]:
data_gdynia["build_street_same_style"] = data_gdynia.groupby(
    ["addr_street", "arch_style"]
).transform("size")

In [ ]:
data_gdynia.loc[:, ["addr_street", "arch_style", "build_street_same_style"]]

,addr_street,arch_style,build_street_same_style
id,,,
1,Świętego Piotra,współczesność,8.0
2,Świętego Piotra,współczesność,8.0
3,Węglowa,współczesność,1.0
4,Świętego Piotra,współczesność,8.0
5,Tadeusza Wendy,współczesność,9.0
...,...,...,...
1237,Władysława IV,modernizm,24.0
1238,Świętojańska,modernizm,107.0
1239,Aleja Marszałka Józefa Piłsudskiego,modernizm,3.0


In [ ]:
assert (
    data_gdynia["build_street_same_style"].isna().sum()
    == data_gdynia["addr_street"].isna().sum()
)

In [ ]:
data_gdynia.groupby("addr_street").transform("size")

id
1        16.0
2        16.0
3         1.0
4        16.0
5         9.0
        ...  
1237     41.0
1238    125.0
1239      3.0
1240      3.0
1241      3.0
Length: 1105, dtype: float64

In [ ]:
data_gdynia_streets_over_20 = data_gdynia.loc[
    data_gdynia.groupby(["addr_street"]).transform("size") >= 20
]
data_gdynia_streets_over_20.head()

,osm_id,addr_street,addr_number,ground_floor_group,ground_floor_function,upper_floor_group,upper_floor_function,number_of_storeys,is_mixed_use,build_year_est,arch_style,condition,max_height,is_heritage,source,confidence,date_of_checking,number_of_functions,street_mean,storeys_vs_street,mean_street,std_street,z_in_street,order_in_street,build_street_same_style
id,,,,,,,,,,,,,,,,,,,,,,,,,
20,126117726,Stefana Żeromskiego,1,NaN,mieszkaniowa jednorodzinna,NaN,Mieszkaniowa Jednorodzinna,2,<NA>,1950,modernizm,2,NaN,NaN,google.com/maps,2,2026-09-27,1,4.533333,-2.533333,4.533333,1.892676,-1.338493,12.0,20.0
23,127110164,Jerzego Waszyngtona,12,NaN,mieszkaniowa wielorodzinna,NaN,Mieszkaniowa Wielorodzinna,4,<NA>,1950,modernizm,1,NaN,NaN,google.com/maps,2,2026-09-27,1,4.827586,-0.827586,4.827586,2.817054,-0.293777,9.0,7.0
31,244042555,Stefana Żeromskiego,3,NaN,mieszkaniowa jednorodzinna,NaN,Mieszkaniowa Jednorodzinna,2,<NA>,1950,modernizm,2,NaN,NaN,google.com/maps,2,2026-09-27,1,4.533333,-2.533333,4.533333,1.892676,-1.338493,13.0,20.0
32,398139546,Stefana Żeromskiego,NaN,NaN,mieszkaniowa jednorodzinna,NaN,NaN,1,<NA>,1950,modernizm,2,NaN,NaN,google.com/maps,2,2026-09-27,1,4.533333,-3.533333,4.533333,1.892676,-1.866845,14.0,20.0
40,126117721,Stefana Żeromskiego,4,NaN,mieszkaniowa wielorodzinna,NaN,Mieszkaniowa Wielorodzinna,5,<NA>,1940,modernizm,2,NaN,NaN,google.com/maps,2,2026-09-27,1,4.533333,0.466667,4.533333,1.892676,0.246564,15.0,20.0


In [ ]:
assert data_gdynia_streets_over_20["addr_street"].nunique() == 13
assert len(data_gdynia_streets_over_20) == 605